In [1]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
from shapely.geometry import Polygon, Point
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from collections import Counter
import os
import datetime as dt
import networkx as nx
import pickle

In [14]:
def get_roads_within_distance(R, seed_road, max_distance_m):
    """
    Returns all roads reachable from seed_road within max_distance_m meters.

    R                : networkx.Graph where nodes have attribute 'length_m'
    seed_road        : node ID (road_id) from which to start
    max_distance_m   : maximum cumulative distance in meters

    Returns:
      distances : dict {road_id: cumulative_distance}
      df_edges  : pandas.DataFrame with edges between reachable roads and their attributes
    """
    # 1. Ensure each edge has a 'weight' attribute: avg of node lengths
    for u, v, data in R.edges(data=True):
        if 'weight' not in data:
            data['weight'] = (R.nodes[u]['length_m'] + R.nodes[v]['length_m']) / 2.0

    # 2. Run Dijkstra up to cutoff
    distances = nx.single_source_dijkstra_path_length(
        R, seed_road, cutoff=max_distance_m, weight='weight'
    )
    reachable = set(distances.keys())

    return list(reachable)

In [3]:
roads_gdf = gpd.read_file('datos/2_Estructura_via/estructura_via_completa/MVI.geojson')
roads_gdf = roads_gdf.to_crs(epsg = 4326)
roads_gdf["id_mvi"] = roads_gdf["MVICIV"].astype(str) + "_" + roads_gdf["MVISVIA"].str.replace(" ", "")
roads_gdf['contador'] = roads_gdf.groupby('id_mvi').cumcount() + 1
roads_gdf['id_mvi'] = roads_gdf['id_mvi'] + '.' + roads_gdf['contador'].astype(str)
roads_gdf = (
    roads_gdf.explode(ignore_index=True)          # if shapefile has split segments
       .dissolve(by='id_mvi')               # merge all segments of same road
       .reset_index()
)
roads_gdf['road_id'] = roads_gdf['id_mvi']
roads_gdf['buffer'] = roads_gdf.geometry.buffer(0.001)
roads_buf = roads_gdf.set_geometry('buffer')[['road_id','buffer']]

/tmp/ipykernel_19772/4093957458.py:12: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  roads_gdf['buffer'] = roads_gdf.geometry.buffer(0.001)


In [4]:
events = pd.read_csv('datos/8_eventos/events_dataset_preprocessing.csv', encoding = 'latin-1')
events = events.loc[pd.notnull(events.Longitude)]
events['geometry'] = list(map(lambda x,y:  Point(x,y), events['Longitude'], events['Latutude']))
events = gpd.GeoDataFrame(events).set_crs(epsg = 4326)
events['id_event'] = events.index

In [5]:
joined = gpd.sjoin(
    events,
    roads_buf,
    how='left',
    predicate='intersects'
)

In [6]:
joined = joined.drop(['index_right'],axis=1)

In [7]:
file_input_name = "datos_procesados/conectivity_graph/road_graph.pickle"

with open(file_input_name, 'rb') as f:
    R = pickle.load(f)

In [8]:
hop_distances = {'ALTA':1000, 'MEDIA':800, 'BAJA':400, 'SIN DEFINIR':1000}

In [9]:
joined['hop_distance'] = list(tqdm(map(lambda x: hop_distances.get(x,1000), joined.Complejidad)))

0it [00:00, ?it/s]

In [11]:
joined

,tipo_actividad,fecha_inicio,fecha_final,Longitude,Latutude,Complejidad,geometry,id_event,road_id,hop_distance
188,AGLOMERACION,2024-03-30 00:00:00,2024-03-31 00:00:00,-74.036637,4.801258,ALTA,POINT (-74.03664 4.80126),188,1000002_B.1,1000
189,AGLOMERACION,2024-06-29 00:00:00,2024-07-01 00:00:00,-74.036637,4.801258,ALTA,POINT (-74.03664 4.80126),189,1000002_B.1,1000
190,MANIFESTACION,2024-05-26 00:00:00,2024-05-27 00:00:00,-74.037804,4.685022,MEDIA,POINT (-74.0378 4.68502),190,NaN,800
191,EMERGENCIA,2024-11-06 00:00:00,2024-11-07 00:00:00,-74.038606,4.796220,ALTA,POINT (-74.03861 4.79622),191,11012141_FT.1,1000
191,EMERGENCIA,2024-11-06 00:00:00,2024-11-07 00:00:00,-74.038606,4.796220,ALTA,POINT (-74.03861 4.79622),191,11012141_TF.1,1000
...,...,...,...,...,...,...,...,...,...,...
589,EMERGENCIA,2024-10-01 00:00:00,2024-10-02 00:00:00,-74.158666,4.546244,NaN,POINT (-74.15867 4.54624),589,0_SD.1037,1000
589,EMERGENCIA,2024-10-01 00:00:00,2024-10-02 00:00:00,-74.158666,4.546244,NaN,POINT (-74.15867 4.54624),589,19009719_SD.1,1000
590,MANIFESTACION,2024-06-09 00:00:00,2024-06-10 00:00:00,-74.176305,4.628002,MEDIA,POINT (-74.1763 4.628),590,7001355_B.1,800
590,MANIFESTACION,2024-06-09 00:00:00,2024-06-10 00:00:00,-74.176305,4.628002,MEDIA,POINT (-74.1763 4.628),590,8005300_B.1,800


In [15]:
joined['reachable_nodes'] = list(tqdm(map(lambda road_id, hop_distance: get_roads_within_distance(R, road_id, hop_distance)if pd.notnull(road_id) else [], joined['road_id'], joined['hop_distance'] )))

0it [00:00, ?it/s]

In [13]:
joined

,tipo_actividad,fecha_inicio,fecha_final,Longitude,Latutude,Complejidad,geometry,id_event,road_id,hop_distance,reachable_nodes
188,AGLOMERACION,2024-03-30 00:00:00,2024-03-31 00:00:00,-74.036637,4.801258,ALTA,POINT (-74.03664 4.80126),188,1000002_B.1,1000,"{1006568_B.1, 1000003_B.1, 11012471_B.1, 10062..."
189,AGLOMERACION,2024-06-29 00:00:00,2024-07-01 00:00:00,-74.036637,4.801258,ALTA,POINT (-74.03664 4.80126),189,1000002_B.1,1000,"{1006568_B.1, 1000003_B.1, 11012471_B.1, 10062..."
190,MANIFESTACION,2024-05-26 00:00:00,2024-05-27 00:00:00,-74.037804,4.685022,MEDIA,POINT (-74.0378 4.68502),190,NaN,800,[]
191,EMERGENCIA,2024-11-06 00:00:00,2024-11-07 00:00:00,-74.038606,4.796220,ALTA,POINT (-74.03861 4.79622),191,11012141_FT.1,1000,"{11012142_FT.1, 11012674_B.1, 11012142_TF.3, 1..."
191,EMERGENCIA,2024-11-06 00:00:00,2024-11-07 00:00:00,-74.038606,4.796220,ALTA,POINT (-74.03861 4.79622),191,11012141_TF.1,1000,"{11000040_FT.1, 11000058_B.1, 11000042_B.1, 11..."
...,...,...,...,...,...,...,...,...,...,...,...
589,EMERGENCIA,2024-10-01 00:00:00,2024-10-02 00:00:00,-74.158666,4.546244,NaN,POINT (-74.15867 4.54624),589,0_SD.1037,1000,"{19009347_B.1, 19010235_B.1, 19008858_B.1, 190..."
589,EMERGENCIA,2024-10-01 00:00:00,2024-10-02 00:00:00,-74.158666,4.546244,NaN,POINT (-74.15867 4.54624),589,19009719_SD.1,1000,"{19009347_B.1, 19010235_B.1, 19008858_B.1, 190..."
590,MANIFESTACION,2024-06-09 00:00:00,2024-06-10 00:00:00,-74.176305,4.628002,MEDIA,POINT (-74.1763 4.628),590,7001355_B.1,800,"{7001310_B.1, 0_FT.12, 7001106_B.1, 8005567_TF..."
590,MANIFESTACION,2024-06-09 00:00:00,2024-06-10 00:00:00,-74.176305,4.628002,MEDIA,POINT (-74.1763 4.628),590,8005300_B.1,800,"{7001440_B.1, 8005095_TF.3, 0_FT.12, 8004538_B..."


In [ ]:
joined

In [ ]:
joined = joined.loc[pd.notnull(joined.index_right)].reset_index()
joined['totals'] =1
joined['closing_date'] = list(tqdm(map(lambda x: dt.datetime.strptime(x, '%Y-%m-%d %H:%M:%S'), joined['Fecha Cierre'])))
joined['timeframe'] = joined['closing_date'].dt.floor("15min")

In [ ]:
incident_type_df = joined.groupby(['road_id','timeframe','incident_type']).agg({'totals':'sum'}).reset_index().pivot_table(index = ['road_id','timeframe'], columns = 'incident_type', values= 'totals').fillna(0).reset_index()
incident_type_df.columns  = [f'incident_type_{col}' if col not in ['road_id','timeframe'] else col for col in incident_type_df.columns]

In [ ]:
case_specifics_df = joined.groupby(['road_id','timeframe','case_specifics']).agg({'totals':'sum'}).reset_index().pivot_table(index = ['road_id','timeframe'], columns = 'case_specifics', values= 'totals').fillna(0).reset_index()
case_specifics_df.columns  = [f'incident_case_specifics_{col}' if col not in ['road_id','timeframe'] else col for col in case_specifics_df.columns]

In [ ]:
incidents_data = pd.merge(incident_type_df, case_specifics_df, on = ['road_id','timeframe'], how = 'left')

In [ ]:
file_output_name = "datos_procesados/incidents/"
os.makedirs(file_output_name, exist_ok=True)
incidents_data.to_csv(f'{file_output_name}incidents_dataset.csv', index=False)